# Find, download, and process a NISAR GUNW with SnowIn

Install the development notebook stack from the checkout with `python -m pip install -e ".[dev,gunw,notebooks]"`.

This example uses `snowin.io.find_nisar` and `snowin.io.download_urls`, thin wrappers around the public `nisar_pytools` search and validated downloader. It then opens one GUNW with SnowIn, checks the normalized phase, adds local-incidence geometry, and computes pairwise dSWE.

The search can find GSLC and other NISAR product types, but SnowIn's current snow retrieval workflow consumes GUNW. This notebook therefore searches for GUNW. It selects one matching product by default because a full GUNW can be several gigabytes. ASF download access may require an Earthdata Login configured in your local environment. For Zach Hoppinen's full NIVAL comparison, see [`05_nival_nisar_comparison.ipynb`](05_nival_nisar_comparison.ipynb); this notebook demonstrates discovery and SnowIn processing without repeating that study.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, download_urls, find_nisar, open_gunw

aoi = [-115, 43, -114, 44]  # [west, south, east, north]
start_date = "2025-08-01"
end_date = "2025-12-01"
product_type = "GUNW"  # SnowIn's current phase retrieval adapter consumes GUNW
out_dir = Path.home() / ".cache" / "snowin" / "nisar_search_example"
max_results = 20
max_products = 1  # Raise only when you intend to download more large products

## Search ASF

`find_nisar` returns a list of download URLs. Narrow by track and direction if those matter for your time series; the optional filters follow `nisar_pytools.find_nisar`.

In [ ]:
urls = find_nisar(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    product_type=product_type,
    path_number=77,
    direction="ASCENDING",
    max_results=max_results,
)

print(f"Found {len(urls)} matching {product_type} products")
for index, url in enumerate(urls[:10]):
    print(f"{index}: {url.rsplit('/', 1)[-1]}")

In [ ]:
if not urls:
    raise RuntimeError("No GUNW products matched. Adjust the AOI, dates, or filters.")

selected_urls = urls[:max_products]
files = download_urls(selected_urls, out_dir, max_workers=2, validate=True)
if not files:
    raise RuntimeError("No products were downloaded and validated. Check ASF access and try again.")

gunw_path = Path(files[0])
print(f"Using {gunw_path}")

## Open with SnowIn and inspect the canonical phase

SnowIn uses `nisar_pytools.open_nisar` to read the GUNW DataTree, then returns a normalized `xarray.Dataset`. It includes the canonical phase, coherence, connected components, ionospheric phase and uncertainty, and the full hydrostatic/wet tropospheric screens on their separate native radar grid. These correction layers are exposed for inspection and are not applied automatically.

The `pair` dataset owns a lazy file handle; keep it open while using its arrays and close it when finished. The incidence step can download/cache DEM tiles and reads the GUNW radar-grid LOS data.

In [ ]:
pair = open_gunw(gunw_path, chunks="auto")
display(pair)
print("phase definition:", pair.phase.attrs["phase_difference_definition"])
print("reference → secondary:", pair.attrs["reference_time"], "→", pair.attrs["secondary_time"])
print("wavelength (m):", pair.attrs["wavelength_m"])

In [ ]:
plot_layers = list(pair.data_vars)
ncols = 3
nrows = max(1, (len(plot_layers) + ncols - 1) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.8 * nrows), squeeze=False)

for ax, name in zip(axes.flat, plot_layers):
    layer = pair[name]
    title = name.replace("_", " ").title()
    if "radar_height" in layer.dims:
        layer = layer.sel(radar_height=0, method="nearest")
        selected_height = float(layer.radar_height.values)
        title += f" (radar height {selected_height:g} m)"

    y_dim = "radar_y" if "radar_y" in layer.dims else "y"
    x_dim = "radar_x" if "radar_x" in layer.dims else "x"
    if name == "connected_component":
        preview = layer.isel({y_dim: slice(None, None, 8), x_dim: slice(None, None, 8)})
        cmap = "tab20"
    else:
        preview = layer.coarsen({y_dim: 8, x_dim: 8}, boundary="trim").mean()
        cmap = "RdBu_r" if name == "phase" else ("plasma" if name == "coherence" else "magma")
    preview.compute().plot.imshow(ax=ax, cmap=cmap, robust=True)
    ax.set_title(title)

for ax in axes.flat[len(plot_layers):]:
    fig.delaxes(ax)

fig.suptitle("SnowIn GUNW layers (correction screens are shown, not applied)")
fig.tight_layout()
plt.show()


## Add local incidence and calculate pairwise dSWE

The default local-incidence path uses SnowIn's NISAR-modified Copernicus DEM adapter. Set `dem_cache_dir` to control where DEM assets are cached, or supply a local DEM using `nisar_cop30_dem=...`. The resulting dSWE is pairwise change, not absolute SWE.

In [ ]:
pair = add_gunw_incidence(
    pair,
    gunw_path,
    dem_cache_dir=out_dir / "dem_cache",
)
print(f"geometry-supported fraction: {float(pair.geometry_valid.mean()):.6f}")
dswe = compute_dswe(
    pair.phase,
    pair.incidence_angle,
    wavelength_m=pair.attrs["wavelength_m"],
)
display(dswe)
print("dSWE units:", dswe.attrs["units"])

In [ ]:
dswe_preview = dswe.coarsen(y=8, x=8, boundary="trim").mean().compute()
dswe_preview.plot(cmap="BrBG", robust=True, figsize=(9, 6))
plt.title("Pairwise dSWE (m water equivalent)")
plt.show()
pair.close()